# 05 — Threshold and PCA extensions

**Purpose.** Study validation-only classification thresholds and run the PCA Logistic Regression extension.

**Inputs.** Validation predictions, training data, and feature dictionary from earlier notebooks.

**Outputs.** Threshold tables and figure, PCA validation results, PCA loadings, and PCA comparison figure.

**What you will learn.** How threshold choices affect precision and recall, and why PCA is treated as an extension rather than the main model.

**Run first.** Run Notebooks 01-04 first.

## Imports and paths

In [1]:
from __future__ import annotations

import ast
import json
import math
from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator, PercentFormatter
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_sample_weight

current_path = Path.cwd().resolve()
if current_path.name == "notebooks":
    project_root = current_path.parent
else:
    project_root = current_path

assert project_root.name == "ml-finance-bankruptcy-analysis", (
    f"Unexpected project root: {project_root}"
)

DATA_DIR = project_root / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
OUTPUTS_DIR = project_root / "outputs"
FIGURES_DIR = OUTPUTS_DIR / "figures"
TABLES_DIR = OUTPUTS_DIR / "tables"
REPORT_DIR = OUTPUTS_DIR / "report"
PAPER_TABLES_DIR = OUTPUTS_DIR / "paper" / "tables"
PAPER_FIGURES_DIR = OUTPUTS_DIR / "paper" / "figures"

for path in [
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    FIGURES_DIR,
    TABLES_DIR,
    REPORT_DIR,
    PAPER_TABLES_DIR,
    PAPER_FIGURES_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)


## Project constants

In [2]:
RAW_DATA_PATH = RAW_DATA_DIR / "american_bankruptcy.csv"
MODEL_DATASET_PATH = PROCESSED_DATA_DIR / "model_dataset.csv"
TRAIN_DATA_PATH = PROCESSED_DATA_DIR / "train.csv"
TEST_DATA_PATH = PROCESSED_DATA_DIR / "test.csv"

COMPANY_COLUMN = "company_name"
RAW_TARGET_COLUMN = "status_label"
TARGET_COLUMN = "failed"
YEAR_COLUMN = "year"

ALIVE_LABEL = "alive"
FAILED_LABEL = "failed"

RANDOM_STATE = 42
OUTER_TEST_SIZE = 0.20
VALIDATION_SIZE = 0.20
LOGISTIC_C_GRID = [0.01, 0.1, 1.0, 10.0]
PCA_COMPONENT_GRID = [2, 3, 5, 8, 10, 12, 15, 18]

EXPECTED_MODEL_ORDER = [
    "Majority-class baseline",
    "Logistic Regression",
    "L1 Regularized Logistic Regression",
    "L2 Regularized Logistic Regression",
    "Decision Tree",
    "Random Forest",
    "Gradient Boosting",
]

PREDICTION_TABLE_COLUMNS = [
    "model",
    COMPANY_COLUMN,
    YEAR_COLUMN,
    "actual_failed",
    "predicted_failed",
    "probability_failed",
]
PREDICTION_PROBABILITY_FLOAT_FORMAT = "%.11f"


## Feature names and descriptions

In [3]:
FEATURE_NAME_MAP = {
    "X1": "Current assets",
    "X2": "Cost of goods sold",
    "X3": "Depreciation and amortization",
    "X4": "EBITDA",
    "X5": "Inventory",
    "X6": "Net income",
    "X7": "Total receivables",
    "X8": "Market value",
    "X9": "Net sales",
    "X10": "Total assets",
    "X11": "Total long-term debt",
    "X12": "EBIT",
    "X13": "Gross profit",
    "X14": "Total current liabilities",
    "X15": "Retained earnings",
    "X16": "Total revenue",
    "X17": "Total liabilities",
    "X18": "Total operating expenses",
}

FEATURE_DESCRIPTION_MAP = {
    "X1": "Assets expected to be sold, converted into cash, or used within one year.",
    "X2": "Direct costs related to producing or selling the firm's goods and services.",
    "X3": "Depreciation of tangible assets and amortization of intangible assets.",
    "X4": "Earnings before interest, taxes, depreciation, and amortization.",
    "X5": "Goods and raw materials held by the firm for production or sale.",
    "X6": "Profit after expenses and costs have been deducted from revenue.",
    "X7": "Money owed to the firm by customers for delivered goods or services.",
    "X8": "Market capitalization or market value of the publicly traded company.",
    "X9": "Gross sales minus returns, allowances, and discounts.",
    "X10": "Total assets owned or controlled by the company.",
    "X11": "Debt obligations due after more than one year.",
    "X12": "Earnings before interest and taxes.",
    "X13": "Profit after subtracting costs related to manufacturing and selling.",
    "X14": "Short-term obligations due within one year.",
    "X15": "Accumulated profits retained in the business after dividends and losses.",
    "X16": "Total income from sales before subtracting expenses.",
    "X17": "Total debts and obligations owed to outside parties.",
    "X18": "Expenses incurred through normal business operations.",
}

KEY_FEATURES_FOR_EDA = ["X8", "X6", "X11", "X1", "X17", "X15"]
KEY_MODELS_FOR_THRESHOLD_ANALYSIS = [
    "Logistic Regression",
    "Random Forest",
    "Gradient Boosting",
]


## Figure style helpers

In [4]:
MODEL_COLORS = {
    "Majority-class baseline": "#8c8c8c",
    "Logistic Regression": "#4c78a8",
    "Random Forest": "#59a14f",
    "Gradient Boosting": "#f28e2b",
}
OUTCOME_COLORS = {"detected": "#59a14f", "missed": "#e15759", "false_alarm": "#f28e2b"}
DIRECTION_COLORS = {"positive": "#c44e52", "negative": "#4c78a8"}
METRIC_COLORS = {
    "Precision": "#4c78a8",
    "Recall": "#e15759",
    "F1": "#59a14f",
    "ROC-AUC": "#4c78a8",
    "PR-AUC": "#f28e2b",
    "Failed F1": "#59a14f",
}
BASELINE_COLOR = "#8c8c8c"


def apply_project_style() -> None:
    """Apply the common Matplotlib style used across project figures."""
    plt.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "figure.facecolor": "white",
            "axes.facecolor": "white",
            "axes.titlesize": 12,
            "axes.labelsize": 10,
            "xtick.labelsize": 9,
            "ytick.labelsize": 9,
            "legend.fontsize": 8.5,
            "axes.spines.top": False,
            "axes.spines.right": False,
            "grid.color": "#d9d9d9",
            "grid.linewidth": 0.8,
            "lines.linewidth": 2.0,
            "lines.markersize": 5.5,
        }
    )


def style_axis(ax, *, grid_axis: str = "y", percent_y: bool = False, integer_x: bool = False) -> None:
    """Add consistent grid and tick formatting to one axis."""
    ax.grid(axis=grid_axis, linestyle="--", alpha=0.25)
    if percent_y:
        ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
    if integer_x:
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))


def save_figure(fig, output_path: Path) -> None:
    """Save a Matplotlib figure with the project's standard export settings."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


## Data validation and feature helpers

In [5]:
def validate_required_columns(data: pd.DataFrame) -> None:
    """Confirm that the raw dataset has the identifier, year, and target columns."""
    missing = {COMPANY_COLUMN, RAW_TARGET_COLUMN, YEAR_COLUMN}.difference(data.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")


def validate_target_labels(data: pd.DataFrame) -> None:
    """Confirm that the raw target contains only the expected alive/failed labels."""
    observed = set(data[RAW_TARGET_COLUMN].dropna().unique())
    unexpected = observed.difference({ALIVE_LABEL, FAILED_LABEL})
    if unexpected:
        raise ValueError(f"Unexpected target labels found: {sorted(unexpected)}")


def identify_feature_columns(data: pd.DataFrame) -> list[str]:
    """Return numeric financial variables from the raw dataset, excluding identifiers."""
    excluded = {COMPANY_COLUMN, RAW_TARGET_COLUMN, YEAR_COLUMN}
    return [
        column
        for column in data.columns
        if column not in excluded and pd.api.types.is_numeric_dtype(data[column])
    ]


def get_feature_columns(data: pd.DataFrame, include_year: bool = False) -> list[str]:
    """Return modelling predictors while keeping identifiers and target out of X."""
    excluded = {COMPANY_COLUMN, TARGET_COLUMN}
    if not include_year:
        excluded.add(YEAR_COLUMN)
    return [column for column in data.columns if column not in excluded]


def split_features_target(data: pd.DataFrame, include_year: bool = False) -> tuple[pd.DataFrame, pd.Series]:
    """Split a model-ready table into predictor matrix and binary target."""
    feature_columns = get_feature_columns(data, include_year=include_year)
    return data[feature_columns].copy(), data[TARGET_COLUMN].copy()


## Company split helpers

In [6]:
def create_company_level_split(
    data: pd.DataFrame,
    test_size: float = OUTER_TEST_SIZE,
    random_state: int = RANDOM_STATE,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Split rows by company so a firm cannot appear in both samples."""
    missing = {COMPANY_COLUMN, TARGET_COLUMN}.difference(data.columns)
    if missing:
        raise ValueError(f"Missing required split columns: {sorted(missing)}")

    splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_idx, test_idx = next(
        splitter.split(data, y=data[TARGET_COLUMN], groups=data[COMPANY_COLUMN])
    )
    return data.iloc[train_idx].copy(), data.iloc[test_idx].copy()


def check_no_company_overlap(left: pd.DataFrame, right: pd.DataFrame) -> bool:
    """Return True when two samples have disjoint company identifiers."""
    left_companies = set(left[COMPANY_COLUMN].unique())
    right_companies = set(right[COMPANY_COLUMN].unique())
    return left_companies.isdisjoint(right_companies)


def create_split_summary(
    full_data: pd.DataFrame,
    train_data: pd.DataFrame,
    test_data: pd.DataFrame,
) -> pd.DataFrame:
    """Summarize row counts, company counts, and failure rates for a split."""
    def summarize(name: str, data: pd.DataFrame) -> dict[str, float | int | str]:
        return {
            "split": name,
            "n_rows": int(len(data)),
            "n_companies": int(data[COMPANY_COLUMN].nunique()),
            "n_failed_rows": int(data[TARGET_COLUMN].sum()),
            "n_alive_rows": int(len(data) - data[TARGET_COLUMN].sum()),
            "failure_rate": float(data[TARGET_COLUMN].mean()),
        }

    summary = pd.DataFrame(
        [summarize("full", full_data), summarize("train", train_data), summarize("test", test_data)]
    )
    summary["company_share"] = summary["n_companies"] / int(full_data[COMPANY_COLUMN].nunique())
    summary["row_share"] = summary["n_rows"] / int(len(full_data))
    return summary


## Evaluation helpers

In [7]:
def get_probability_failed(model, x: pd.DataFrame) -> np.ndarray:
    """Return predicted probabilities for the failed class."""
    return model.predict_proba(x)[:, 1]


def evaluate_binary_classifier(
    model_name: str,
    y_true: pd.Series,
    y_pred: np.ndarray,
    probability_failed: np.ndarray,
) -> dict[str, float | int | str]:
    """Calculate the common classification metrics for one fitted model."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, probability_failed),
        "pr_auc": average_precision_score(y_true, probability_failed),
        "precision_failed": precision_score(y_true, y_pred, zero_division=0),
        "recall_failed": recall_score(y_true, y_pred, zero_division=0),
        "f1_failed": f1_score(y_true, y_pred, zero_division=0),
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp),
    }


def create_classification_report_table(
    model_name: str,
    y_true: pd.Series,
    y_pred: np.ndarray,
) -> pd.DataFrame:
    """Convert scikit-learn's classification report into a flat table."""
    report = classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=["alive", "failed"],
        output_dict=True,
        zero_division=0,
    )
    rows = []
    for label, metrics in report.items():
        if isinstance(metrics, dict):
            rows.append({"model": model_name, "class_or_average": label, **metrics})
    return pd.DataFrame(rows)


def create_prediction_table(
    data: pd.DataFrame,
    model_name: str,
    y_pred: np.ndarray,
    probability_failed: np.ndarray,
) -> pd.DataFrame:
    """Create an identifier-preserving prediction table for one model."""
    table = pd.DataFrame(
        {
            "model": model_name,
            COMPANY_COLUMN: data[COMPANY_COLUMN].to_numpy(),
            YEAR_COLUMN: data[YEAR_COLUMN].to_numpy(),
            "actual_failed": data[TARGET_COLUMN].to_numpy(),
            "predicted_failed": y_pred,
            "probability_failed": probability_failed,
        }
    )
    return table[PREDICTION_TABLE_COLUMNS]


def save_prediction_table(predictions: pd.DataFrame, output_path: Path) -> None:
    """Write prediction probabilities with deterministic decimal formatting."""
    predictions[PREDICTION_TABLE_COLUMNS].to_csv(
        output_path,
        index=False,
        float_format=PREDICTION_PROBABILITY_FLOAT_FORMAT,
    )


## Model builders and selection helpers

In [8]:
def build_majority_class_baseline() -> DummyClassifier:
    """Build the benchmark model that always predicts the most frequent class."""
    return DummyClassifier(strategy="most_frequent")


def build_logistic_pipeline(
    penalty: str = "l2",
    c_value: float = 1.0,
    class_weight: str | dict | None = "balanced",
) -> Pipeline:
    """Build a scaled Logistic Regression pipeline with the original settings."""
    if penalty not in {"l1", "l2"}:
        raise ValueError("Only 'l1' and 'l2' penalties are supported in this project.")

    l1_ratio = 1.0 if penalty == "l1" else 0.0
    model = LogisticRegression(
        C=c_value,
        l1_ratio=l1_ratio,
        solver="saga",
        class_weight=class_weight,
        max_iter=5_000,
        random_state=RANDOM_STATE,
    )
    return Pipeline(steps=[("scaler", StandardScaler()), ("model", model)])


def build_interpretable_logit() -> Pipeline:
    """Build the fixed Logistic Regression benchmark used for interpretation."""
    return build_logistic_pipeline(penalty="l2", c_value=1.0)


def select_regularized_logit(
    x_train: pd.DataFrame,
    y_train: pd.Series,
    x_valid: pd.DataFrame,
    y_valid: pd.Series,
    penalty: str,
    c_grid: list[float],
) -> tuple[Pipeline, float, float]:
    """Select L1 or L2 Logistic Regression by validation PR-AUC."""
    best_model, best_c, best_score = None, None, -1.0
    for c_value in c_grid:
        candidate = build_logistic_pipeline(penalty=penalty, c_value=c_value)
        candidate.fit(x_train, y_train)
        score = average_precision_score(y_valid, candidate.predict_proba(x_valid)[:, 1])
        if score > best_score:
            best_model, best_c, best_score = candidate, c_value, score
    if best_model is None or best_c is None:
        raise RuntimeError("No Logistic Regression model was selected.")
    return best_model, best_c, best_score


def build_decision_tree(max_depth: int | None = 3, min_samples_leaf: int = 100) -> DecisionTreeClassifier:
    """Build a class-weighted Decision Tree with the source project settings."""
    return DecisionTreeClassifier(
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )


def build_random_forest(
    n_estimators: int = 300,
    max_depth: int | None = 5,
    min_samples_leaf: int = 50,
    max_features: str | None = "sqrt",
) -> RandomForestClassifier:
    """Build a Random Forest using the original class weights and seed."""
    return RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )


def build_gradient_boosting(
    n_estimators: int = 150,
    learning_rate: float = 0.05,
    max_depth: int = 2,
    min_samples_leaf: int = 100,
) -> GradientBoostingClassifier:
    """Build a Gradient Boosting classifier with the source project settings."""
    return GradientBoostingClassifier(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        random_state=RANDOM_STATE,
    )


def select_decision_tree(
    x_train: pd.DataFrame,
    y_train: pd.Series,
    x_valid: pd.DataFrame,
    y_valid: pd.Series,
) -> tuple[DecisionTreeClassifier, dict[str, int], float]:
    """Search the original Decision Tree grid and keep the best PR-AUC model."""
    grid = {"max_depth": [2, 3, 4, 5], "min_samples_leaf": [50, 100, 200]}
    best_model, best_params, best_score = None, None, -1.0
    for max_depth, min_samples_leaf in product(grid["max_depth"], grid["min_samples_leaf"]):
        candidate = build_decision_tree(max_depth=max_depth, min_samples_leaf=min_samples_leaf)
        candidate.fit(x_train, y_train)
        score = average_precision_score(y_valid, candidate.predict_proba(x_valid)[:, 1])
        if score > best_score:
            best_model = candidate
            best_params = {"max_depth": max_depth, "min_samples_leaf": min_samples_leaf}
            best_score = score
    if best_model is None or best_params is None:
        raise RuntimeError("No Decision Tree model was selected.")
    return best_model, best_params, best_score


def select_random_forest(
    x_train: pd.DataFrame,
    y_train: pd.Series,
    x_valid: pd.DataFrame,
    y_valid: pd.Series,
) -> tuple[RandomForestClassifier, dict[str, object], float]:
    """Search the original Random Forest grid and keep the best PR-AUC model."""
    grid = {
        "n_estimators": [300],
        "max_depth": [4, 6, None],
        "min_samples_leaf": [50, 100],
        "max_features": ["sqrt"],
    }
    best_model, best_params, best_score = None, None, -1.0
    for n_estimators, max_depth, min_samples_leaf, max_features in product(
        grid["n_estimators"], grid["max_depth"], grid["min_samples_leaf"], grid["max_features"]
    ):
        candidate = build_random_forest(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
        )
        candidate.fit(x_train, y_train)
        score = average_precision_score(y_valid, candidate.predict_proba(x_valid)[:, 1])
        if score > best_score:
            best_model = candidate
            best_params = {
                "n_estimators": n_estimators,
                "max_depth": max_depth,
                "min_samples_leaf": min_samples_leaf,
                "max_features": max_features,
            }
            best_score = score
    if best_model is None or best_params is None:
        raise RuntimeError("No Random Forest model was selected.")
    return best_model, best_params, best_score


def select_gradient_boosting(
    x_train: pd.DataFrame,
    y_train: pd.Series,
    x_valid: pd.DataFrame,
    y_valid: pd.Series,
) -> tuple[GradientBoostingClassifier, dict[str, object], float]:
    """Search the original Gradient Boosting grid using balanced sample weights."""
    grid = {
        "n_estimators": [100, 150],
        "learning_rate": [0.03, 0.05],
        "max_depth": [2, 3],
        "min_samples_leaf": [100],
    }
    sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)
    best_model, best_params, best_score = None, None, -1.0
    for n_estimators, learning_rate, max_depth, min_samples_leaf in product(
        grid["n_estimators"], grid["learning_rate"], grid["max_depth"], grid["min_samples_leaf"]
    ):
        candidate = build_gradient_boosting(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
        )
        candidate.fit(x_train, y_train, sample_weight=sample_weight)
        score = average_precision_score(y_valid, candidate.predict_proba(x_valid)[:, 1])
        if score > best_score:
            best_model = candidate
            best_params = {
                "n_estimators": n_estimators,
                "learning_rate": learning_rate,
                "max_depth": max_depth,
                "min_samples_leaf": min_samples_leaf,
            }
            best_score = score
    if best_model is None or best_params is None:
        raise RuntimeError("No Gradient Boosting model was selected.")
    return best_model, best_params, best_score


## Model reconstruction helpers

In [9]:
def parse_selected_parameters(value: object) -> dict[str, object]:
    """Parse the saved parameter string from `model_specification.csv`."""
    if pd.isna(value) or value == "":
        return {}

    text = str(value)
    if text.startswith("{"):
        return ast.literal_eval(text)

    params: dict[str, object] = {}
    for piece in text.split(","):
        if "=" in piece:
            key, val = piece.strip().split("=", 1)
            params[key] = float(val) if key == "C" else val
    return params


def build_model_from_spec(model_name: str, selected_parameters: object):
    """Recreate one model from its saved specification without using model binaries."""
    params = parse_selected_parameters(selected_parameters)
    if model_name == "Majority-class baseline":
        return build_majority_class_baseline()
    if model_name == "Logistic Regression":
        return build_interpretable_logit()
    if model_name == "L1 Regularized Logistic Regression":
        return build_logistic_pipeline(penalty="l1", c_value=params["C"])
    if model_name == "L2 Regularized Logistic Regression":
        return build_logistic_pipeline(penalty="l2", c_value=params["C"])
    if model_name == "Decision Tree":
        return build_decision_tree(**params)
    if model_name == "Random Forest":
        return build_random_forest(**params)
    if model_name == "Gradient Boosting":
        return build_gradient_boosting(**params)
    raise ValueError(f"Unknown model: {model_name}")


## Load validation predictions

In [10]:
predictions = pd.read_csv(TABLES_DIR / "validation_predictions.csv")
model_counts = predictions.groupby("model").size().reset_index(name="n_predictions")
display(model_counts)

,model,n_predictions
0,Decision Tree,12665
1,Gradient Boosting,12665
2,L1 Regularized Logistic Regression,12665
3,L2 Regularized Logistic Regression,12665
4,Logistic Regression,12665
5,Majority-class baseline,12665
6,Random Forest,12665


## Threshold grid

In [11]:
thresholds = [value / 100 for value in range(1, 100)]
threshold_grid_summary = pd.DataFrame({"first_threshold": [thresholds[0]], "last_threshold": [thresholds[-1]], "step": [thresholds[1] - thresholds[0]], "n_thresholds": [len(thresholds)]})
display(threshold_grid_summary)

,first_threshold,last_threshold,step,n_thresholds
0,0.01,0.99,0.01,99


## Precision, recall, and F1 across thresholds

In [12]:
rows = []
for model_name in KEY_MODELS_FOR_THRESHOLD_ANALYSIS:
    model_predictions = predictions[predictions["model"] == model_name]
    y_true = model_predictions["actual_failed"]
    probability_failed = model_predictions["probability_failed"]
    for threshold in thresholds:
        y_pred = (probability_failed >= threshold).astype(int)
        rows.append(
            {
                "model": model_name,
                "threshold": threshold,
                "precision_failed": precision_score(y_true, y_pred, zero_division=0),
                "recall_failed": recall_score(y_true, y_pred, zero_division=0),
                "f1_failed": f1_score(y_true, y_pred, zero_division=0),
                "predicted_failed_share": float(y_pred.mean()),
                "n_predicted_failed": int(y_pred.sum()),
                "n_actual_failed": int(y_true.sum()),
            }
        )
threshold_analysis = pd.DataFrame(rows)
assert set(threshold_analysis["model"]) == set(KEY_MODELS_FOR_THRESHOLD_ANALYSIS), "Threshold analysis must use the declared model subset."
assert threshold_analysis["threshold"].between(0.01, 0.99).all(), "Thresholds must stay within original range."
display(threshold_analysis.groupby("model").head(5))

,model,threshold,precision_failed,recall_failed,f1_failed,predicted_failed_share,n_predicted_failed,n_actual_failed
0,Logistic Regression,0.01,0.066817,1.000000,0.125264,0.980813,12422,830
1,Logistic Regression,0.02,0.066984,1.000000,0.125558,0.978366,12391,830
2,Logistic Regression,0.03,0.067049,0.998795,0.125663,0.976234,12364,830
3,Logistic Regression,0.04,0.067147,0.998795,0.125835,0.974812,12346,830
4,Logistic Regression,0.05,0.067186,0.997590,0.125893,0.973075,12324,830
99,Random Forest,0.01,0.066199,1.000000,0.124177,0.989972,12538,830
100,Random Forest,0.02,0.066533,0.997590,0.124746,0.982629,12445,830
101,Random Forest,0.03,0.066882,0.996386,0.125351,0.976313,12365,830
102,Random Forest,0.04,0.067341,0.993976,0.126137,0.967311,12251,830
103,Random Forest,0.05,0.067664,0.986747,0.126643,0.955705,12104,830


## Select threshold scenarios

In [13]:
selected_rows = []
for model_name, model_data in threshold_analysis.groupby("model"):
    best_f1 = model_data.sort_values(["f1_failed", "precision_failed", "threshold"], ascending=[False, False, True]).iloc[0]
    selected_rows.append(
        {
            "model": model_name,
            "selection_rule": "maximize_f1",
            "threshold": best_f1["threshold"],
            "precision_failed": best_f1["precision_failed"],
            "recall_failed": best_f1["recall_failed"],
            "f1_failed": best_f1["f1_failed"],
            "predicted_failed_share": best_f1["predicted_failed_share"],
            "n_predicted_failed": best_f1["n_predicted_failed"],
        }
    )
    for recall_target in (0.6, 0.8):
        feasible = model_data[model_data["recall_failed"] >= recall_target]
        if feasible.empty:
            continue
        best = feasible.sort_values(["precision_failed", "f1_failed", "threshold"], ascending=[False, False, False]).iloc[0]
        selected_rows.append(
            {
                "model": model_name,
                "selection_rule": f"recall_at_least_{recall_target:.1f}",
                "threshold": best["threshold"],
                "precision_failed": best["precision_failed"],
                "recall_failed": best["recall_failed"],
                "f1_failed": best["f1_failed"],
                "predicted_failed_share": best["predicted_failed_share"],
                "n_predicted_failed": best["n_predicted_failed"],
            }
        )
selected_thresholds = pd.DataFrame(selected_rows)
threshold_analysis.to_csv(TABLES_DIR / "validation_threshold_analysis.csv", index=False)
selected_thresholds.to_csv(TABLES_DIR / "selected_thresholds.csv", index=False)
display(selected_thresholds)

,model,selection_rule,threshold,precision_failed,recall_failed,f1_failed,predicted_failed_share,n_predicted_failed
0,Gradient Boosting,maximize_f1,0.68,0.211806,0.220482,0.216057,0.068220,864
1,Gradient Boosting,recall_at_least_0.6,0.48,0.108621,0.613253,0.184554,0.369996,4686
2,Gradient Boosting,recall_at_least_0.8,0.37,0.090046,0.822892,0.162329,0.598895,7585
3,Logistic Regression,maximize_f1,0.53,0.190408,0.320482,0.238886,0.110304,1397
4,Logistic Regression,recall_at_least_0.6,0.52,0.088176,0.681928,0.156159,0.506830,6419
5,Logistic Regression,recall_at_least_0.8,0.50,0.081713,0.845783,0.149029,0.678326,8591
6,Random Forest,maximize_f1,0.57,0.187075,0.265060,0.219342,0.092854,1176
7,Random Forest,recall_at_least_0.6,0.37,0.114961,0.615663,0.193744,0.350967,4445
8,Random Forest,recall_at_least_0.8,0.27,0.099970,0.812048,0.178024,0.532333,6742


## Threshold trade-off figure

In [14]:
apply_project_style()
fig, axes = plt.subplots(1, len(KEY_MODELS_FOR_THRESHOLD_ANALYSIS), figsize=(11.5, 3.8), sharex=True, sharey=True)
for ax, model_name in zip(axes, KEY_MODELS_FOR_THRESHOLD_ANALYSIS, strict=False):
    model_data = threshold_analysis[threshold_analysis["model"] == model_name]
    ax.plot(model_data["threshold"], model_data["precision_failed"], label="Precision", color=METRIC_COLORS["Precision"])
    ax.plot(model_data["threshold"], model_data["recall_failed"], label="Recall", color=METRIC_COLORS["Recall"])
    ax.plot(model_data["threshold"], model_data["f1_failed"], label="F1", color=METRIC_COLORS["F1"])
    selected = selected_thresholds[(selected_thresholds["model"] == model_name) & (selected_thresholds["selection_rule"] == "maximize_f1")]
    if not selected.empty:
        selected_threshold = selected.iloc[0]["threshold"]
        ax.axvline(selected_threshold, color="#4d4d4d", linestyle=":", linewidth=1.4, label="Max-F1 threshold" if ax is axes[0] else None)
        ax.text(selected_threshold + 0.015, 0.05, f"{selected_threshold:.2f}", rotation=90, fontsize=8, color="#4d4d4d")
    ax.set_title(model_name)
    ax.set_ylim(0, 1)
    style_axis(ax, grid_axis="both")
    ax.set_xlabel("Failure-probability threshold")
axes[0].set_ylabel("Metric value")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=4, bbox_to_anchor=(0.5, 0.03))
fig.suptitle("Validation Threshold Trade-off", fontsize=12)
save_figure(fig, FIGURES_DIR / "validation_threshold_tradeoff.png")
assert (FIGURES_DIR / "validation_threshold_tradeoff.png").exists(), "Threshold figure was not saved."


## Load data for PCA extension

In [15]:
train_full = pd.read_csv(TRAIN_DATA_PATH)
feature_dictionary = pd.read_csv(TABLES_DIR / "feature_dictionary.csv")
model_train, validation = create_company_level_split(train_full, test_size=VALIDATION_SIZE, random_state=RANDOM_STATE)
x_train, y_train = split_features_target(model_train)
x_valid, y_valid = split_features_target(validation)
valid_components = sorted({value for value in PCA_COMPONENT_GRID if 1 <= value <= x_train.shape[1]})
assert valid_components == PCA_COMPONENT_GRID, "PCA component grid should be valid for X1-X18."
display(pd.DataFrame({"n_features": [x_train.shape[1]], "candidate_components": [valid_components]}))

,n_features,candidate_components
0,18,"[2, 3, 5, 8, 10, 12, 15, 18]"


## PCA Logistic Regression candidates

In [16]:
def build_pca_logistic_pipeline(n_components: int) -> Pipeline:
    """Build the PCA plus Logistic Regression extension model."""
    return Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("pca", PCA(n_components=n_components, random_state=RANDOM_STATE)),
            ("model", LogisticRegression(C=1.0, l1_ratio=0.0, solver="saga", class_weight="balanced", max_iter=5_000, random_state=RANDOM_STATE)),
        ]
    )

rows = []
best_model, best_pr_auc = None, -1.0
for n_components in valid_components:
    model_name = f"PCA Logistic Regression ({n_components} components)"
    model = build_pca_logistic_pipeline(n_components)
    model.fit(x_train, y_train)
    y_pred = model.predict(x_valid)
    probability_failed = get_probability_failed(model, x_valid)
    metrics = evaluate_binary_classifier(model_name, y_valid, y_pred, probability_failed)
    metrics["n_components"] = n_components
    metrics["cumulative_explained_variance"] = float(model.named_steps["pca"].explained_variance_ratio_.sum())
    rows.append(metrics)
    if metrics["pr_auc"] > best_pr_auc:
        best_pr_auc, best_model = metrics["pr_auc"], model

pca_results = pd.DataFrame(rows)
assert pca_results["n_components"].tolist() == PCA_COMPONENT_GRID, "PCA output must preserve component order."
pca_results.to_csv(TABLES_DIR / "pca_logistic_results.csv", index=False)
display(pca_results)

,model,accuracy,balanced_accuracy,roc_auc,pr_auc,precision_failed,recall_failed,f1_failed,true_negative,false_positive,false_negative,true_positive,n_components,cumulative_explained_variance
0,PCA Logistic Regression (2 components),0.257718,0.531130,0.497775,0.063412,0.070376,0.845783,0.129940,2562,9273,128,702,2,0.838036
1,PCA Logistic Regression (3 components),0.355310,0.580547,0.642485,0.115707,0.079849,0.839759,0.145831,3803,8032,133,697,3,0.891845
2,PCA Logistic Regression (5 components),0.356336,0.587258,0.651072,0.124425,0.081025,0.853012,0.147993,3805,8030,122,708,5,0.939105
3,PCA Logistic Regression (8 components),0.360600,0.592340,0.669184,0.139993,0.082011,0.859036,0.149727,3854,7981,117,713,8,0.979586
4,PCA Logistic Regression (10 components),0.365417,0.600519,0.671400,0.146718,0.083555,0.871084,0.152483,3905,7930,107,723,10,0.991918
5,PCA Logistic Regression (12 components),0.369601,0.592675,0.673717,0.150720,0.082321,0.849398,0.150096,3976,7859,125,705,12,0.998211
6,PCA Logistic Regression (15 components),0.366917,0.589558,0.670708,0.147184,0.081704,0.845783,0.149013,3945,7890,128,702,15,1.000000
7,PCA Logistic Regression (18 components),0.366917,0.589558,0.670708,0.147184,0.081704,0.845783,0.149013,3945,7890,128,702,18,1.000000


## PCA component loadings

In [17]:
pca = best_model.named_steps["pca"]
loadings = pd.DataFrame(pca.components_, columns=get_feature_columns(model_train), index=[f"PC{i}" for i in range(1, pca.n_components_ + 1)])
pca_component_loadings = loadings.reset_index(names="principal_component").melt(id_vars="principal_component", var_name="feature", value_name="loading")
pca_component_loadings["absolute_loading"] = pca_component_loadings["loading"].abs()
pca_component_loadings = pca_component_loadings.merge(feature_dictionary[["feature", "readable_name", "description"]], on="feature", how="left").sort_values(["principal_component", "absolute_loading"], ascending=[True, False])
pca_component_loadings.to_csv(TABLES_DIR / "pca_component_loadings.csv", index=False)
display(pca_component_loadings.groupby("principal_component").head(5))

,principal_component,feature,loading,absolute_loading,readable_name,description
180,PC1,X16,0.255537,0.255537,Total revenue,Total income from sales before subtracting exp...
96,PC1,X9,0.255537,0.255537,Net sales,"Gross sales minus returns, allowances, and dis..."
36,PC1,X4,0.255067,0.255067,EBITDA,"Earnings before interest, taxes, depreciation,..."
108,PC1,X10,0.254127,0.254127,Total assets,Total assets owned or controlled by the company.
156,PC1,X14,0.254117,0.254117,Total current liabilities,Short-term obligations due within one year.
81,PC10,X7,0.578393,0.578393,Total receivables,Money owed to the firm by customers for delive...
165,PC10,X14,-0.549436,0.549436,Total current liabilities,Short-term obligations due within one year.
93,PC10,X8,0.292874,0.292874,Market value,Market capitalization or market value of the p...
57,PC10,X5,0.229215,0.229215,Inventory,Goods and raw materials held by the firm for p...
33,PC10,X3,0.207351,0.207351,Depreciation and amortization,Depreciation of tangible assets and amortizati...


## PCA metric figure

In [18]:
apply_project_style()
fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.2), sharex=True, width_ratios=[1, 1])
axes[0].plot(pca_results["n_components"], pca_results["roc_auc"], marker="o", label="ROC-AUC", color=METRIC_COLORS["ROC-AUC"])
axes[1].plot(pca_results["n_components"], pca_results["pr_auc"], marker="o", label="PR-AUC", color=METRIC_COLORS["PR-AUC"])
axes[1].plot(pca_results["n_components"], pca_results["f1_failed"], marker="o", label="Failed F1", color=METRIC_COLORS["Failed F1"])
axes[0].set_title("Ranking performance")
axes[0].set_ylabel("ROC-AUC")
axes[0].set_ylim(0.45, 0.75)
axes[1].set_title("Minority-class operating metrics")
axes[1].set_ylabel("Metric value")
axes[1].set_ylim(0.04, 0.20)
for ax in axes:
    ax.set_xlabel("Number of principal components")
    style_axis(ax, grid_axis="both", integer_x=True)
    ax.legend(loc="best")
fig.suptitle("PCA Validation Performance", fontsize=12)
save_figure(fig, FIGURES_DIR / "pca_logistic_metric_comparison.png")
assert (FIGURES_DIR / "pca_logistic_metric_comparison.png").exists(), "PCA figure was not saved."
